# Replication Goal

The Federal Reserve reports that 63% of adults could cover a $400
emergency expense using cash or its equivalent.

Our goal is to reproduce this estimate from the 2025 SHED public-use
microdata before performing preprocessing or model development.

Declared tolerance: ±0.5 percentage points.

In [2]:
import os
import pandas as pd
import numpy as np

DATA_PATH = os.path.join(
    os.getcwd(),
    "..",
    "data",
    "raw",
    "SHED 2025 CSV Raw.csv"
)

df = pd.read_csv(DATA_PATH, low_memory=False)

In [4]:
df.shape

(12934, 815)

In [5]:
required_columns = [
    "shedid",
    "weight",
    "weight_pop",
    "pay_casheqv"
]

for col in required_columns:
    print(col, col in df.columns)

shedid True
weight True
weight_pop True
pay_casheqv True


In [6]:
df["pay_casheqv"].value_counts(dropna=False)

pay_casheqv
Yes    8398
No     4536
Name: count, dtype: int64

In [7]:
df["pay_casheqv"].isna().sum()

np.int64(0)

`pay_casheqv` is the derived SHED variable used here to represent whether
a respondent could cover the $400 emergency expense using cash or its equivalent.

In [8]:
unweighted_percent = (
    (df["pay_casheqv"] == "Yes").mean() * 100
)

print(f"Unweighted estimate: {unweighted_percent:.2f}%")

Unweighted estimate: 64.93%


In [9]:
yes_mask = df["pay_casheqv"] == "Yes"

yes_weight = df.loc[
    yes_mask,
    "weight_pop"
].sum()

total_weight = df["weight_pop"].sum()

weighted_percent = (
    yes_weight / total_weight
) * 100

print(f"Weighted estimate: {weighted_percent:.2f}%")

Weighted estimate: 63.14%


In [10]:
FEDERAL_ESTIMATE = 63.0
TOLERANCE = 0.5

absolute_error = abs(
    weighted_percent - FEDERAL_ESTIMATE
)

print(f"Federal Reserve estimate: {FEDERAL_ESTIMATE:.2f}%")
print(f"Our weighted estimate: {weighted_percent:.2f}%")
print(f"Absolute error: {absolute_error:.2f} percentage points")

Federal Reserve estimate: 63.00%
Our weighted estimate: 63.14%
Absolute error: 0.14 percentage points


In [11]:
replication_passed = absolute_error <= TOLERANCE

print(
    "Replication result:",
    "PASS" if replication_passed else "FAIL"
)

Replication result: PASS


In [12]:
weighted_percent_sample = (
    df.loc[
        yes_mask,
        "weight"
    ].sum()
    /
    df["weight"].sum()
) * 100

print(
    f"Using weight: {weighted_percent_sample:.2f}%"
)

print(
    f"Using weight_pop: {weighted_percent:.2f}%"
)

Using weight: 63.14%
Using weight_pop: 63.14%


## Replication Result

Published Federal Reserve estimate:
63.0%

Our weighted estimate:
63.14%

Absolute error:
0.14 percentage points

Tolerance:
±0.5 percentage points

Result:
PASS

Conclusion:
The 2025 SHED dataset, target variable, and survey-weighting procedure are
consistent with the published Federal Reserve estimate. The project can proceed
to preprocessing and model development.